# Decision Trees & Ensembles — Titanic SurvivalWe compare a single decision tree, a random forest, and gradient boosting on the **Titanic** dataset (predict whether a passenger survived).**Models:**- **Decision Tree** — recursive splits maximizing information gain- **Random Forest** — bag of decorrelated trees (variance reduction)- **Gradient Boosting** — sequential trees, each correcting predecessors (bias reduction)**Dataset:** 891 passengers, demographic + ticket features. Source: well-known Kaggle / Stanford CS109 mirror.

In [ ]:
import osimport urllib.requestDATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data") if os.path.basename(os.getcwd()) == "notebooks" else "data"os.makedirs(DATA_DIR, exist_ok=True)def download_if_needed(url, filename):    """Download a CSV if it doesn't already exist locally."""    path = os.path.join(DATA_DIR, filename)    if not os.path.exists(path):        print(f"Downloading {filename} from {url}")        urllib.request.urlretrieve(url, path)    return pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.tree import DecisionTreeClassifier, plot_treefrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.metrics import accuracy_score, classification_reportsns.set_style("whitegrid")np.random.seed(42)

## 1. Load data

In [ ]:
path = download_if_needed(    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",    "titanic.csv",)df = pd.read_csv(path)print("Shape:", df.shape)df.head()

## 2. Light cleaning + feature engineering

In [ ]:
work = df.copy()# Encode sex, fill missing age with median, drop high-cardinality / leaky colswork["Sex"] = (work["Sex"] == "female").astype(int)  # 1 if femalework["Age"] = work["Age"].fillna(work["Age"].median())work["Embarked"] = work["Embarked"].fillna("S")work = pd.get_dummies(work, columns=["Embarked"], drop_first=True)feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",                "Embarked_Q", "Embarked_S"]work = work[feature_cols + ["Survived"]].dropna()print("Cleaned shape:", work.shape)work.head()

## 3. Quick EDA: survival by sex and class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))df.groupby("Sex")["Survived"].mean().plot(    kind="bar", ax=axes[0], color=["steelblue", "salmon"])axes[0].set_title("Survival rate by sex"); axes[0].set_ylabel("survival rate")df.groupby("Pclass")["Survived"].mean().plot(    kind="bar", ax=axes[1], color="seagreen")axes[1].set_title("Survival rate by passenger class")axes[1].set_ylabel("survival rate")plt.tight_layout(); plt.show()

Sex and class are by far the strongest predictors — women and first-class passengers were much more likely to survive.

## 4. Split

In [ ]:
X = work[feature_cols].valuesy = work["Survived"].valuesX_tr, X_te, y_tr, y_te = train_test_split(    X, y, test_size=0.25, random_state=42, stratify=y)

## 5. Single decision tree (depth 3 — for visualization)

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)print(f"Test accuracy: {accuracy_score(y_te, tree.predict(X_te)):.4f}")plt.figure(figsize=(16, 7))plot_tree(tree, feature_names=feature_cols,          class_names=["died", "survived"], filled=True, rounded=True)plt.tight_layout(); plt.show()

## 6. Compare three models

In [ ]:
models = {    "Decision Tree":     DecisionTreeClassifier(max_depth=5, random_state=42),    "Random Forest":     RandomForestClassifier(n_estimators=200, random_state=42),    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),}rows = []for name, m in models.items():    m.fit(X_tr, y_tr)    cv = cross_val_score(m, X_tr, y_tr, cv=5).mean()    test_acc = accuracy_score(y_te, m.predict(X_te))    rows.append({"Model": name, "CV acc": cv, "Test acc": test_acc})pd.DataFrame(rows).round(4)

## 7. Feature importances (Random Forest)

In [ ]:
rf = models["Random Forest"]imp = pd.DataFrame({    "feature": feature_cols,    "importance": rf.feature_importances_,}).sort_values("importance")imp.plot.barh(x="feature", y="importance",              figsize=(8, 5), legend=False, color="seagreen")plt.title("Random Forest feature importances")plt.tight_layout(); plt.show()

## Takeaways- A simple depth-3 tree already captures the major patterns (Sex → Pclass → Age).- **Random Forest** and **Gradient Boosting** push test accuracy to ~83%, comfortably above the single tree.- Random Forest = bagging + decorrelated trees (each sees a random subset of features). Reduces variance.- Gradient Boosting = sequential trees, each fitting the residuals of its predecessors. Reduces bias.- Tree-based methods give feature importances "for free" — useful for interpretation. They also handle mixed feature types and missing values gracefully (no scaling needed).